## Read-only results viewer -- SAFE TO RUN WHILE OTHER NOTEBOOKS ARE RUNNING

**This notebook computes nothing and writes nothing.** Every cell loads already-saved
files (`metrics.json`, `umap_cells.csv`, `*_per_batch.csv`,
`rare_affinity_purity_{ds}.json`) and displays them. No training, no clustering, no
affinity graphs, no files written -- so it cannot interfere with a run in progress in
another Colab session.

Use it to look at whatever is finished so far. Anything still running simply does not
appear yet; re-run the cells later to pick it up.

# Results so far: scProto vs. Harmony at d=8 / d=20 / d=50

Shows every (dataset, dimension) combination present on disk:

- **scProto** at latent_dims 8 (published), 20, 50
- **Harmony** at 8, 20, 50 PCs, each with SEACells and Leiden on top
- **SEACells (PCA)** -- the paper's own uncorrected baseline

Metrics are the paper's own. The **rare-cell table** is the one to read first: coverage,
recall, precision, homogeneity, cross-batch homogeneity and macro F1, each mean +- std
across that dataset's batches.

## Setup

In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
numba 0.66.0 requires numpy<2.5,>=1.22, but you have numpy 2.5.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requi

  Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
Using cached numpy-2.2.6-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.5 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=

In [1]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [2]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [3]:
import os
import numpy as np
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir, get_seacell_model_dir
from interpretable_ssl.evaluation.metric_helpers.result_tables import extract_model_key
from interpretable_ssl.evaluation.rebuttal_report import (
    build_model_keywords, dim_matched_read_only_keywords,
    RARE_CELL_METRICS, RARE_CELL_SIG_METRICS, SCPROTO_KEY,
)
from interpretable_ssl.evaluation.paper_figures import (
    rare_celltype_purity_table, rare_metric_significance_paired,
    graph_batch_significance_paired, graph_batch_significance,
    _resolve_run_dir, _read_series,
)

print("imports ready")

imports ready


## Config

In [4]:
ALL_DATASETS = ['pancreas', 'lung', 'pbmc-immune']
DIMS = [8, 20, 50]        # every dimension to look for
REF_DIM = 20              # the dimension whose same-dimension comparison is shown below

dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

## What exists on disk right now

Pure directory listing. `harmony` is present when that dimension's SEACells run has saved
metrics; `scproto` when a run folder for that latent dimension has both metrics and
umap_cells (everything the tables below need).

In [5]:
def scproto_dirs(ds_id, latent_dim):
    """Run folders for scProto at `latent_dim`. d=8 is the published run, whose folder
    carries no LD token (latent_dims only appears in the name when it differs from the
    default 8) -- matched by the canonical key instead."""
    base = get_dataset_model_dir(ds_id)
    if not os.path.isdir(base):
        return []
    if latent_dim == 8:
        return sorted(d for d in os.listdir(base)
                      if os.path.isdir(os.path.join(base, d))
                      and 'LD' not in d
                      and extract_model_key(d, ds_id=ds_id) == SCPROTO_KEY)
    token = f'LD{latent_dim}'
    return sorted(d for d in os.listdir(base)
                  if token in d and d.startswith('proto_umap')
                  and os.path.isdir(os.path.join(base, d)))


def scproto_ready(ds_id, latent_dim):
    for d in scproto_dirs(ds_id, latent_dim):
        p = os.path.join(get_dataset_model_dir(ds_id), d)
        if (os.path.exists(os.path.join(p, 'metrics.json'))
                and os.path.exists(os.path.join(p, 'umap_cells.csv'))):
            return True
    return False


def harmony_ready(ds_id, dim):
    return os.path.exists(os.path.join(get_seacell_model_dir(ds_id, f'X_harmony_d{dim}'),
                                       'metrics.json'))


rows = []
for ds_id in ALL_DATASETS:
    for d in DIMS:
        rows.append({'dataset': ds_id, 'dim': d,
                     'harmony': 'yes' if harmony_ready(ds_id, d) else '-',
                     'scproto': 'yes' if scproto_ready(ds_id, d) else '-'})
df_inventory = pd.DataFrame(rows).set_index(['dataset', 'dim'])
display(df_inventory)

# Only datasets with at least the published scProto run are worth tabulating -- it is the
# reference every table below compares against.
DATASETS_TO_SHOW = [ds for ds in ALL_DATASETS if scproto_ready(ds, 8)]
print(f"\ndatasets included below: {DATASETS_TO_SHOW}")

harmony scproto
dataset     dim                
pancreas    8       yes     yes
            20      yes     yes
            50      yes     yes
lung        8       yes     yes
            20      yes     yes
            50      yes     yes
pbmc-immune 8       yes     yes
            20        -       -
            50      yes     yes


datasets included below: ['pancreas', 'lung', 'pbmc-immune']


## Method rows

One display name per (method, dimension) so no dimension can silently collapse into
another. Rows whose runs are missing simply never match anything and are dropped by the
tables themselves.

In [6]:
read_only = {}
for d in DIMS:
    read_only.update(dim_matched_read_only_keywords('harmony', f'Harmony d={d}', matched_dim=d))

# build_model_keywords adds scProto (published d=8) + SEACells (PCA); correction_methods
# left empty because this notebook computes nothing -- every Harmony row comes in as a
# read-only keyword above.
MODEL_KEYWORDS = build_model_keywords([], {}, extra_read_only=read_only)

for d in DIMS:
    if d == 8:
        continue   # published run already present as 'scProto'
    keys = {extract_model_key(x, ds_id=ds)
            for ds in DATASETS_TO_SHOW for x in scproto_dirs(ds, d)}
    if keys:
        MODEL_KEYWORDS[sorted(keys)[0]] = f'scProto (d={d})'

MODEL_KEYWORDS

{'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto',
 'seacell': 'SEACells (PCA)',
 'seacell_X_harmony_d8': 'SEACells (Harmony d=8)',
 'leiden_X_harmony_d8': 'Leiden (Harmony d=8)',
 'seacell_X_harmony_d20': 'SEACells (Harmony d=20)',
 'leiden_X_harmony_d20': 'Leiden (Harmony d=20)',
 'seacell_X_harmony_d50': 'SEACells (Harmony d=50)',
 'leiden_X_harmony_d50': 'Leiden (Harmony d=50)',
 'proto_umap_LD20_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto (d=20)',
 'proto_umap_LD50_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto (d=50)'}

## Rare-cell metrics

The key table. Read F1 together with its two halves: precision and recall move in
opposite directions between scProto and Harmony, and cross-batch homogeneity is the
metric Harmony's batch-mixing objective most directly targets.

In [7]:
df_rare = rare_celltype_purity_table(DATASETS_TO_SHOW, model_keywords=MODEL_KEYWORDS,
                                     verbose=False, quiet=True)

dupe = df_rare.index.duplicated(keep='first')
if dupe.any():
    print(f"dropped {dupe.sum()} duplicate row(s): {df_rare.index[dupe].tolist()}")
    df_rare = df_rare[~dupe]

show_table(df_rare, metrics=RARE_CELL_METRICS, dataset_display_names=dataset_display_names)

In [8]:
# Same numbers as a plain frame (easier to copy exact values than off the styled table).
df_rare[[c for c in RARE_CELL_METRICS if c in df_rare.columns]].round(3)

batch_rare_coverage_mean  \
dataset     run                                                 
pancreas    scProto                                      0.64   
            SEACells (PCA)                               0.43   
            SEACells (Harmony d=8)                       0.56   
            Leiden (Harmony d=8)                         0.40   
            SEACells (Harmony d=20)                      0.69   
            Leiden (Harmony d=20)                        0.60   
            SEACells (Harmony d=50)                      0.79   
            Leiden (Harmony d=50)                        0.56   
            scProto (d=20)                               0.76   
            scProto (d=50)                               0.76   
lung        scProto                                      0.79   
            SEACells (PCA)                               0.52   
            SEACells (Harmony d=8)                       0.54   
            Leiden (Harmony d=8)                         0.43   
            SEACells (Harmony d=20)                      0.79   
            Leiden (Harmony d=20)                        0.59   
            SEACells (Harmony d=50)                      0.81   
            Leiden (Harmony d=50)                        0.76   
            scProto (d=20)                               0.70   
            scProto (d=50)                               0.71   
pbmc-immune scProto                                      0.93   
            SEACells (PCA)                               1.00   
            SEACells (Harmony d=8)                       0.93   
            Leiden (Harmony d=8)                         0.93   
            SEACells (Harmony d=50)                      1.00   
            Leiden (Harmony d=50)                        1.00   
            scProto (d=50)                               1.00   

                                     batch_rare_recall_macro_mean  \
dataset     run                                                     
pancreas    scProto                                          0.54   
            SEACells (PCA)                                   0.41   
            SEACells (Harmony d=8)                           0.35   
            Leiden (Harmony d=8)                             0.28   
            SEACells (Harmony d=20)                          0.62   
            Leiden (Harmony d=20)                            0.49   
            SEACells (Harmony d=50)                          0.76   
            Leiden (Harmony d=50)                            0.43   
            scProto (d=20)                                   0.67   
            scProto (d=50)                                   0.62   
lung        scProto                                          0.60   
            SEACells (PCA)                                   0.41   
            SEACells (Harmony d=8)                           0.38   
            Leiden (Harmony d=8)                             0.31   
            SEACells (Harmony d=20)                          0.61   
            Leiden (Harmony d=20)                            0.46   
            SEACells (Harmony d=50)                          0.67   
            Leiden (Harmony d=50)                            0.66   
            scProto (d=20)                                   0.55   
            scProto (d=50)                                   0.59   
pbmc-immune scProto                                          0.88   
            SEACells (PCA)                                   0.87   
            SEACells (Harmony d=8)                           0.77   
            Leiden (Harmony d=8)                             0.80   
            SEACells (Harmony d=50)                          0.89   
            Leiden (Harmony d=50)                            0.75   
            scProto (d=50)                                   0.93   

                                     batch_rare_precision_macro_mean  \
dataset     run                                                        
pancr

### Significance -- scProto (published, d=8) as reference

Paired one-sided Wilcoxon signed-rank on the per-batch values, Bonferroni-corrected per
dataset. `wins=x/n` is how many batches the reference beats that method outright, which
stays readable even when p_adj is 'ns'.

In [9]:
sig_vs_d8 = rare_metric_significance_paired(
    df_rare, ref_name='scProto', metrics=RARE_CELL_SIG_METRICS,
    dataset_display_names=dataset_display_names,
)
sig_vs_d8.round(4)

,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.4377,0.5354,0.2080,NaN,NaN,NaN,NaN
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.3726,0.3706,0.2509,6.0,0.0391,0.3516,ns
2,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=8),220,8,0.2955,0.3165,0.1594,7.0,0.0117,0.1055,ns
3,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=8),220,8,0.1334,0.2236,0.1712,8.0,0.0039,0.0352,*
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=20),220,8,0.5383,0.5372,0.2058,4.0,0.5273,1.0000,ns
...,...,...,...,...,...,...,...,...,...,...,...,...
76,Immune,batch_rare_cross_batch_homog,SEACells (Harmony d=8),300,5,0.3046,0.2454,0.1534,3.0,0.1562,0.9375,ns
77,Immune,batch_rare_cross_batch_homog,Leiden (Harmony d=8),300,5,0.3728,0.3046,0.1818,2.0,0.2326,1.0000,ns
78,Immune,batch_rare_cross_batch_homog,SEACells (Harmony d=50),300,5,0.4111,0.3165,0.1584,3.0,0.4062,1.0000,ns
79,Immune,batch_rare_cross_batch_homog,Leiden (Harmony d=50),300,5,0.5462,0.4542,0.2291,0.0,1.0000,1.0000,ns


### Significance -- scProto at the matched dimension as reference

Same test with `scProto (d={REF_DIM})` as the reference, i.e. comparing each method
against scProto at its own dimension. Empty if that run is not on disk yet.

In [10]:
ref_dim_name = f'scProto (d={REF_DIM})'
if ref_dim_name in set(df_rare.index.get_level_values(1)):
    sig_vs_ref = rare_metric_significance_paired(
        df_rare, ref_name=ref_dim_name, metrics=RARE_CELL_SIG_METRICS,
        dataset_display_names=dataset_display_names,
    )
    display(sig_vs_ref.round(4))
else:
    print(f"'{ref_dim_name}' not present yet -- run its training cell in the d={REF_DIM} "
          f"notebook first.")

,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,scProto,219,8,0.4377,0.5354,0.2080,6.0,0.0391,0.3516,ns
1,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.3726,0.3706,0.2509,6.0,0.0140,0.1260,ns
2,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=8),220,8,0.2955,0.3165,0.1594,8.0,0.0039,0.0352,*
3,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=8),220,8,0.1334,0.2236,0.1712,8.0,0.0039,0.0352,*
4,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=20),220,8,0.5383,0.5372,0.2058,6.0,0.0742,0.6680,ns
5,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=20),220,8,0.4181,0.4807,0.2333,6.0,0.0977,0.8789,ns
6,Pancreas,batch_rare_f1_macro,SEACells (Harmony d=50),220,8,0.5974,0.6583,0.1307,4.0,0.5781,1.0000,ns
7,Pancreas,batch_rare_f1_macro,Leiden (Harmony d=50),220,8,0.3683,0.4148,0.2181,7.0,0.0117,0.1055,ns
8,Pancreas,batch_rare_f1_macro,scProto (d=20),220,8,0.6405,0.6452,0.1787,NaN,NaN,NaN,NaN
9,Pancreas,batch_rare_f1_macro,scProto (d=50),220,8,0.6374,0.6022,0.1806,5.0,0.0641,0.5769,ns


## Table 1 -- community structure / batch integration

Cell-type purity, batch entropy, modularity per batch, coverage.

In [11]:
from interpretable_ssl.evaluation.rebuttal_report import _keep_and_rename_runs

df_task1 = load_task1_multi(DATASETS_TO_SHOW, metrics=TASK1_METRICS)
df_task1 = _keep_and_rename_runs(df_task1, MODEL_KEYWORDS)
show_table(df_task1, metrics=TASK1_METRICS, dataset_display_names=dataset_display_names)

In [12]:
df_task2 = load_task1_multi(DATASETS_TO_SHOW, metrics=TASK2_METRICS)
df_task2 = _keep_and_rename_runs(df_task2, MODEL_KEYWORDS)
show_table(df_task2, metrics=TASK2_METRICS, dataset_display_names=dataset_display_names)

### Modularity significance

Paired Wilcoxon on per-batch modularity (the only pairable Table 1 metric -- a batch is a
shared unit across methods, a metacell is not), plus the unpaired Mann-Whitney version
that also covers purity and batch entropy.

In [13]:
mod_paired = graph_batch_significance_paired(
    DATASETS_TO_SHOW, MODEL_KEYWORDS, ref_name='scProto',
    dataset_display_names=dataset_display_names,
)
sub = mod_paired.copy()
sub['cell'] = sub.apply(
    lambda r: f"{r['median']:.3f} (n={r['n']}) [ref]" if r['method'] == 'scProto'
    else f"{r['median']:.3f} (wins={r.get('n_wins', '?')}/{r['n']}) {r.get('sig', '?')} "
         f"p_adj={r.get('p_adj', float('nan')):.3g}",
    axis=1,
)
display(sub.pivot(index='method', columns='dataset', values='cell'))

dataset,Immune,Lung,Pancreas
method,,,
Leiden (Harmony d=20),NaN,0.525 (wins=16.0/16) *** p_adj=0.000137,0.392 (wins=9.0/9) * p_adj=0.0176
Leiden (Harmony d=50),0.421 (wins=5.0/5) ns p_adj=0.188,0.573 (wins=15.0/16) *** p_adj=0.000687,0.532 (wins=9.0/9) * p_adj=0.0176
Leiden (Harmony d=8),0.250 (wins=5.0/5) ns p_adj=0.188,0.403 (wins=16.0/16) *** p_adj=0.000137,0.614 (wins=6.0/9) ns p_adj=1
SEACells (Harmony d=20),NaN,0.561 (wins=16.0/16) *** p_adj=0.000137,0.494 (wins=8.0/9) ns p_adj=0.0879
SEACells (Harmony d=50),0.356 (wins=5.0/5) ns p_adj=0.188,0.487 (wins=16.0/16) *** p_adj=0.000137,0.419 (wins=9.0/9) * p_adj=0.0176
SEACells (Harmony d=8),0.551 (wins=5.0/5) ns p_adj=0.188,0.625 (wins=14.0/16) ** p_adj=0.00192,0.566 (wins=7.0/9) ns p_adj=1
SEACells (PCA),0.569 (wins=5.0/5) ns p_adj=0.188,0.671 (wins=9.0/16) ns p_adj=1,0.658 (wins=0.0/9) ns p_adj=1
scProto,0.620 (n=5) [ref],0.669 (n=16) [ref],0.621 (n=9) [ref]
scProto (d=20),NaN,0.676 (wins=5.0/16) ns p_adj=1,0.648 (wins=3.0/9) ns p_adj=1


In [14]:
graph_batch_significance(
    DATASETS_TO_SHOW, MODEL_KEYWORDS, ref_name='scProto',
    dataset_display_names=dataset_display_names,
).round(4)

,dataset,metric,method,k,n,median,mean,std,p_vs_ref,p_adj,sig
0,Pancreas,modularity_per_batch,scProto,219,9,0.6215,0.6012,0.0834,NaN,NaN,NaN
1,Pancreas,modularity_per_batch,SEACells (PCA),220,9,0.6579,0.6739,0.0509,0.9740,1.0000,ns
2,Pancreas,modularity_per_batch,SEACells (Harmony d=8),220,9,0.5664,0.5671,0.0177,0.0318,0.2866,ns
3,Pancreas,modularity_per_batch,Leiden (Harmony d=8),220,9,0.6143,0.6125,0.0212,0.3620,1.0000,ns
4,Pancreas,modularity_per_batch,SEACells (Harmony d=20),220,9,0.4944,0.4997,0.0330,0.0108,0.0976,ns
...,...,...,...,...,...,...,...,...,...,...,...
76,Immune,batch_entropy_per_mc,SEACells (Harmony d=8),300,300,0.7260,0.6407,0.5112,1.0000,1.0000,ns
77,Immune,batch_entropy_per_mc,Leiden (Harmony d=8),300,300,1.0711,0.9471,0.3827,1.0000,1.0000,ns
78,Immune,batch_entropy_per_mc,SEACells (Harmony d=50),300,300,0.8538,0.6867,0.4963,1.0000,1.0000,ns
79,Immune,batch_entropy_per_mc,Leiden (Harmony d=50),300,300,-0.0000,0.4984,0.5721,1.0000,1.0000,ns


## Batch entropy per metacell

`frac_single_batch` is the share of metacells holding cells from exactly one batch --
what a median of 0 actually means. Useful for seeing how much batch structure survives
into each method's metacells at each dimension.

In [15]:
rows = []
for ds_id in DATASETS_TO_SHOW:
    for keyword, name in MODEL_KEYWORDS.items():
        run_dir = _resolve_run_dir(ds_id, keyword, prefer_csv='batch_entropy_per_mc.csv')
        if run_dir is None:
            continue
        ser = _read_series(os.path.join(run_dir, 'batch_entropy_per_mc.csv'))
        if ser is None:
            continue
        v = ser.values.astype(float)
        rows.append({'dataset': dataset_display_names.get(ds_id, ds_id), 'method': name,
                     'n_metacells': len(v),
                     'entropy_mean': round(float(np.mean(v)), 3),
                     'entropy_median': round(float(np.median(v)), 3),
                     'frac_single_batch': round(float((v <= 1e-9).mean()), 3)})

pd.DataFrame(rows).sort_values(['dataset', 'entropy_median'], ascending=[True, False])

,dataset,method,n_metacells,entropy_mean,entropy_median,frac_single_batch
23,Immune,Leiden (Harmony d=8),300,0.947,1.071,0.060
24,Immune,SEACells (Harmony d=50),300,0.687,0.854,0.227
22,Immune,SEACells (Harmony d=8),300,0.641,0.726,0.297
20,Immune,scProto,294,0.229,-0.000,0.677
21,Immune,SEACells (PCA),300,0.135,-0.000,0.603
25,Immune,Leiden (Harmony d=50),300,0.498,-0.000,0.540
26,Immune,scProto (d=50),297,0.222,-0.000,0.687
12,Lung,SEACells (Harmony d=8),300,1.520,1.526,0.000
16,Lung,SEACells (Harmony d=50),300,1.386,1.473,0.003
13,Lung,Leiden (Harmony d=8),300,1.340,1.350,0.000


## Embedding-only rare-cell affinity purity

No clustering involved: for each locally-rare-type cell, the fraction of its affinity mass
going to same-type cells, on each embedding directly. The `dim` column separates the
d=8 / d=20 / d=50 rows. Loaded from saved JSON -- nothing is recomputed.

In [16]:
from interpretable_ssl.evaluation.batch_correct_baselines import load_and_compare_affinity_purity

load_and_compare_affinity_purity(DATASETS_TO_SHOW, dataset_display_names=dataset_display_names)

,dataset,method,dim,n,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,Raw PCA (uncorrected),50,8,0.385,0.164,7.0,0.0742,0.5195,ns
1,Pancreas,scProto,8,8,0.454,0.184,NaN,NaN,NaN,NaN
2,Pancreas,scVI (Gaussian),8,8,0.261,0.267,7.0,0.0078,0.0547,ns
3,Pancreas,Harmony,8,8,0.297,0.129,7.0,0.0195,0.1367,ns
4,Pancreas,scPoli (Stage-1),8,8,0.512,0.162,3.0,0.8750,1.0000,ns
5,Pancreas,scVI,8,8,0.365,0.175,8.0,0.0039,0.0273,*
6,Pancreas,Harmony,50,8,0.529,0.176,4.0,0.8086,1.0000,ns
7,Pancreas,Harmony,20,8,0.525,0.177,4.0,0.8750,1.0000,ns
8,Lung,Raw PCA (uncorrected),50,15,0.559,0.194,11.0,0.0240,0.1437,ns
9,Lung,scProto,8,15,0.586,0.208,NaN,NaN,NaN,NaN


## Realized metacell count vs. K

Confirms each Harmony run's downstream clustering landed at that dataset's
`num_prototypes`, so the comparison is at equal K.

In [17]:
from interpretable_ssl.evaluation.batch_correct_baselines import get_realized_seacell_count

rows = []
for ds_id in DATASETS_TO_SHOW:
    k = DATASETS[ds_id]['num_prototypes']
    for d in DIMS:
        tag = f'X_harmony_d{d}'
        n_actual = get_realized_seacell_count(ds_id, tag)
        if n_actual is None:
            continue
        rows.append({'dataset': ds_id, 'method': f'SEACells ({tag})',
                     'n_actual': n_actual, 'target_k': k,
                     'matches_target': abs(n_actual - k) <= 0.05 * k})

df_k = load_task1_multi(DATASETS_TO_SHOW, metrics=['n_clusters', 'resolution'])
if not df_k.empty:
    is_leiden = df_k.index.get_level_values('run').str.startswith('leiden_X_harmony')
    if is_leiden.any():
        out = df_k[is_leiden].copy()
        out['target_k'] = [DATASETS[ds]['num_prototypes'] for ds, _r in out.index]
        out['matches_target'] = out['n_clusters'] == out['target_k']
        display(out)

display(pd.DataFrame(rows).set_index(['dataset', 'method']) if rows else "no SEACells runs found")

n_clusters resolution  target_k  \
dataset     run                                                         
pancreas    leiden_X_harmony_K220          220.0       32.0       220   
            leiden_X_harmony_d20_K220      220.0       32.0       220   
            leiden_X_harmony_d50_K220      220.0       32.0       220   
            leiden_X_harmony_d8_K220       220.0       32.0       220   
lung        leiden_X_harmony_K300          300.0       64.0       300   
            leiden_X_harmony_d20_K300      300.0       64.0       300   
            leiden_X_harmony_d50_K300      300.0       64.0       300   
            leiden_X_harmony_d8_K300       300.0       64.0       300   
pbmc-immune leiden_X_harmony_K119          119.0   7.992236       300   
            leiden_X_harmony_K300          300.0       16.0       300   
            leiden_X_harmony_d50_K300      300.0       16.0       300   
            leiden_X_harmony_d8_K300       300.0       32.0       300   

                                       matches_target  
dataset     run                                        
pancreas    leiden_X_harmony_K220                True  
            leiden_X_harmony_d20_K220            True  
            leiden_X_harmony_d50_K220            True  
            leiden_X_harmony_d8_K220             True  
lung        leiden_X_harmony_K300                True  
            leiden_X_harmony_d20_K300            True  
            leiden_X_harmony_d50_K300            True  
            leiden_X_harmony_d8_K300             True  
pbmc-immune leiden_X_harmony_K119               False  
            leiden_X_harmony_K300                True  
            leiden_X_harmony_d50_K300            True  
            leiden_X_harmony_d8_K300             True

n_actual  target_k  matches_target
dataset     method                                                      
pancreas    SEACells (X_harmony_d8)        220       220            True
            SEACells (X_harmony_d20)       220       220            True
            SEACells (X_harmony_d50)       220       220            True
lung        SEACells (X_harmony_d8)        300       300            True
            SEACells (X_harmony_d20)       300       300            True
            SEACells (X_harmony_d50)       300       300            True
pbmc-immune SEACells (X_harmony_d8)        300       300            True
            SEACells (X_harmony_d50)       300       300            True